# Introdução

A biblioteca RBG foi construída ao longo de um mês, para a disciplina de Machine Learning I, no IMPA Tech. A biblioteca foi feita pelos seguintes alunos:
- Carlos Eduardo Rocha de Paula
- Cristiane Magarinos Sampaio
- Giovani Firmino Mota
- Lisandra Menezes Fagundes
- Lucas Fraga Damasceno
- Lucca Moulin Cruz
- Theo Veiga Drummond Ambrósio

Para mais informações ou para compreender sua estrutura e principais funções implementadas, você pode visitar nosso projeto em https://github.com/fragadmsc/rbg

Neste relatório, serão detalhados como foram feitos os experimentos de regressão, em que utilizamos o dataset de preços de aluguel em Munique (Rent99), e os de classificação, em que utilizamos um dataset que cataloga diversas amostras de feijão em sete espécies diferentes (Dry Bean). As redes neurais foram aplicadas aos dois problemas e entram na comparação junto com os outros modelos.

Ao final da aplicação de diferentes modelos, serão discutidos quais os métodos mais eficazes, não só em relação à sua taxa de acerto nos dados de teste, mas também em relação à sua interpretabilidade, replicabilidade e eficiência computacional.

Este relatório é uma síntese: os detalhes de cada etapa estão nos notebooks da pasta `notebooks/`, e as tabelas abaixo são lidas dos arquivos que eles gravam em `resultados/`, sem treinar nenhum modelo de novo.

| notebook | conteúdo |
|---|---|
| [01_eda_rent99](notebooks/01_eda_rent99.ipynb) | análise exploratória do Rent99, PCA e K-means |
| [02_regressao_rent99](notebooks/02_regressao_rent99.ipynb) | linear, ridge, lasso, polinomial, árvores, bagging e random forest |
| [03_eda_dry_bean](notebooks/03_eda_dry_bean.ipynb) | análise exploratória do Dry Bean, PCA e K-means |
| [04_classificacao_dry_bean](notebooks/04_classificacao_dry_bean.ipynb) | logística, LDA e QDA (com e sem PCA), KNN com PCA, árvores, bagging e random forest |
| [05_redes_neurais](notebooks/05_redes_neurais.ipynb) | MLP de classificação (Dry Bean) e de regressão (Rent99) |

In [ ]:
from pathlib import Path

import altair as alt
import pandas as pd
from IPython.display import display

alt.renderers.enable("png", scale_factor=1.5)

RESULTADOS = Path("resultados")


def ler(*nomes):
    # junta as tabelas gravadas pelos notebooks; avisa se algum ainda não foi executado
    tabelas = []
    for nome in nomes:
        caminho = RESULTADOS / f"{nome}.csv"
        if caminho.exists():
            tabelas.append(pd.read_csv(caminho))
        else:
            print(f"{caminho} não existe: execute o notebook que o gera.")
    return pd.concat(tabelas) if tabelas else pd.DataFrame()

# Dados

| base | problema | linhas | variáveis | alvo |
|---|---|---|---|---|
| Rent99 (pacote R `gamlss.data`) | regressão | 3.082 | 7 usadas (área, ano, localização, banheiro, cozinha, aquecimento, distrito) | `rent`, aluguel mensal em euros |
| Dry Bean (UCI, id 602) | classificação | 13.611 (13.543 sem duplicatas) | 16 medidas de tamanho e forma do grão | `Class`, sete tipos de feijão |

Os CSVs brutos estão em `data/`, junto com o script que os baixa das fontes originais e o dicionário das variáveis.

## O que a análise exploratória mostrou

**Rent99.** O aluguel tem cauda à direita (média de €459 e mediana de €427, máximo de €1.843). A área é a variável mais associada ao aluguel (correlação de 0,585), e o ano de construção tem associação fraca e não linear: os imóveis das décadas de 1980 e 1990 têm aluguéis medianos mais altos. Localização excelente, banheiro e cozinha premium têm aluguéis medianos bem maiores, mas são grupos pequenos (78, 191 e 131 imóveis). As variáveis são pouco correlacionadas entre si, então o PCA precisa de 5 das 6 componentes para chegar a 90% da variância, e o K-means só separa os imóveis pelas variáveis categóricas.

**Dry Bean.** As 16 variáveis são muito correlacionadas (várias medem o tamanho do grão), e 4 componentes principais já guardam 95% da variância. As classes são desbalanceadas (de 522 BOMBAY a 3.546 DERMASON). No PCA e no K-means, BOMBAY fica isolada, enquanto os pares DERMASON × SIRA e CALI × BARBUNYA se sobrepõem; a previsão era que esses seriam os pares mais confundidos pelos classificadores, o que se confirmou.

# Pré-processamento e protocolo

Para que os modelos pudessem ser comparados, tudo o que não é o modelo ficou fixo em `notebooks/utils/dados.py`:

- **Semente** `SEED = 67` em toda divisão e em todo modelo com sorteio.
- **Rent99**: a coluna `rentsqm` é removida, porque é `rent / area` e vazaria o alvo. A divisão é 80% treino (2.465 imóveis) e 20% teste (617).
- **Dry Bean**: as 68 linhas duplicadas são removidas e a divisão é estratificada, com 20% de cada classe no teste (10.831 de treino e 2.712 de teste).
- **Padronização, PCA e escolha de variáveis** são sempre ajustados só no treino e depois aplicados ao teste.
- **Escolha de hiperparâmetros** só com o treino (LOOCV, K-Fold, StratifiedKFold ou uma parte de validação separada do treino). O teste é usado uma vez, no fim.
- **Métricas**: RMSE, MAE e R² na regressão; acurácia, acurácia balanceada (média do recall das classes) e F1 macro na classificação, porque as classes são desbalanceadas.

# Resultados

## Regressão (Rent99), no teste de 617 imóveis

In [ ]:
regressao = ler("regressao", "redes_regressao").sort_values("RMSE").reset_index(drop=True)
regressao.round(3)

In [ ]:
alt.Chart(regressao).mark_bar().encode(
    x=alt.X("RMSE:Q", title="RMSE no teste (euros)"),
    y=alt.Y("modelo:N", sort="x", title=None),
    tooltip=["modelo", alt.Tooltip("RMSE:Q", format=".1f"), alt.Tooltip("R2:Q", format=".3f")],
).properties(width=480, height=28 * len(regressao), title="Regressão: RMSE no teste (menor é melhor)")

Na nossa execução, o polinomial de grau 2 teve o menor RMSE (146,3), seguido da rede neural (147,7) e dos modelos linear, lasso e ridge, empatados em 148,8. Os modelos de árvore vieram depois: bagging 151,0, random forest 154,8, árvore com poda 156,0 e árvore do GridSearch 161,6. Com 2.465 imóveis e poucas variáveis, a relação com o aluguel é quase linear e modelos mais flexíveis ganham pouco. Em todos os modelos o R² fica perto de 0,5 e o erro vem principalmente dos aluguéis mais altos.

## Classificação (Dry Bean), no teste de 2.712 grãos

In [ ]:
classificacao = ler("classificacao", "redes_classificacao").sort_values("acuracia", ascending=False).reset_index(drop=True)
classificacao.round(4)

In [ ]:
longo = classificacao.melt(id_vars="modelo", value_vars=["acuracia", "acuracia_balanceada", "f1_macro"],
                           var_name="métrica", value_name="valor")
alt.Chart(longo).mark_point(filled=True, size=70).encode(
    x=alt.X("valor:Q", title="Valor no teste", scale=alt.Scale(zero=False), axis=alt.Axis(format="%")),
    y=alt.Y("modelo:N", sort=list(classificacao["modelo"]), title=None),
    color=alt.Color("métrica:N", title="Métrica"),
    tooltip=["modelo", "métrica", alt.Tooltip("valor:Q", format=".2%")],
).properties(width=480, height=28 * len(classificacao), title="Classificação: métricas no teste")

Na nossa execução, a rede neural teve a maior acurácia (93,1%), seguida do KNN (92,5%), da QDA com PCA de 99% (91,9%), da logística (91,7%), da QDA com as 16 variáveis (91,7%) e da LDA (91,1%). As árvores, treinadas em 1.500 grãos e com poucas árvores, ficaram perto de 89–90%, e o random forest com 3 árvores em 75,5%. Os erros de todos os modelos se concentram nos pares DERMASON × SIRA e CALI × BARBUNYA, como a análise exploratória antecipava, e BOMBAY é acertada quase sempre.

# Desempenho, interpretabilidade e custo

- **Desempenho**: entre os melhores modelos as diferenças são pequenas: 2,5 euros de RMSE entre o polinomial e o linear e cerca de 1,4 ponto de acurácia entre a rede neural e a logística. A rede neural fica entre os melhores nos dois problemas, mas sem grande vantagem.
- **Interpretabilidade**: a regressão linear e a logística têm coeficientes de leitura direta. Uma árvore podada pode ser lida como regras. Bagging, random forest, KNN e redes neurais não oferecem essa leitura.
- **Custo**: os modelos lineares, LDA e QDA treinam em segundos. O KNN é barato para treinar, mas levou cerca de 40 s para prever o teste. As árvores da `rbg` são as mais caras: no Rent99, o GridSearch da árvore levou cerca de 5 minutos e o do bagging cerca de 4, e no Dry Bean foi preciso usar uma subamostra.
- **Replicabilidade**: todos os notebooks usam a mesma semente, a mesma divisão e os mesmos dados de `data/`, então os números podem ser reproduzidos executando os notebooks em ordem.

# Conclusão

A biblioteca `rbg` foi suficiente para todo o fluxo do projeto: pré-processamento, redução de dimensionalidade, agrupamento, validação cruzada, busca de hiperparâmetros e modelos supervisionados de vários tipos. Nos dois problemas, os modelos simples ficaram a poucos pontos dos mais flexíveis: o linear a 2,5 euros do melhor RMSE e a logística a 1,4 ponto da melhor acurácia. Por isso eles são a nossa recomendação quando é preciso explicar o modelo. O principal ponto a melhorar na biblioteca é a velocidade das árvores e dos conjuntos de árvores, que limitou os experimentos.